[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/04_data_preprocessing/04_pipelines_and_column_transformer.ipynb)

# Bài 04 — Pipeline và ColumnTransformer: tiền xử lý không rò rỉ

**🎯 Mục tiêu**
- Ghép tiền xử lý + model thành **một object** `Pipeline` — fit một lần, không leakage, dễ triển khai.
- Xử lý riêng cột số và cột phân loại bằng `ColumnTransformer`.
- Viết **custom transformer** theo chuẩn scikit-learn.
- Dùng pipeline với cross-validation, tìm hyperparameter cho cả bước tiền xử lý, lưu/nạp pipeline.

```text
Pipeline([
    ("preprocess", ColumnTransformer([...])),
    ("model", ...)
])
```

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/04_data_preprocessing

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("../../data/sample/house_prices_dirty.csv").drop_duplicates().reset_index(drop=True)
df["district"] = df["district"].str.strip().str.lower().str.replace("q.", "quận", regex=False).str.title()
X = df.drop(columns="price_billion_vnd")
y = np.log(df["price_billion_vnd"])                   # log target: giá lệch phải
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.head()

## 1. Vấn đề khi làm "thủ công"

Không có pipeline, ta phải nhớ: fit imputer trên train → transform test; fit scaler trên train → transform test; fit encoder... Dễ quên, dễ sai thứ tự, và khi **cross-validation**, các bước này phải được fit lại **trong từng fold** — gần như chắc chắn sẽ rò rỉ nếu làm tay.

## 2. ColumnTransformer: mỗi nhóm cột một cách xử lý

In [ ]:
numeric_features = ["area_m2", "bedrooms", "bathrooms", "age_years", "distance_to_center_km"]
categorical_features = ["district"]
boolean_features = ["has_garden"]

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
    ("bool", "passthrough", boolean_features),
])

model = Pipeline([
    ("preprocess", preprocess),
    ("model", Ridge(alpha=1.0)),
])
model

Chỉ cần **một** lệnh `fit` — mọi bước học tham số từ `X_train`:

In [ ]:
model.fit(X_train, y_train)
pred = model.predict(X_test)                         # imputer/scaler/encoder dùng tham số của train
print(f"MAE (log giá) trên test: {mean_absolute_error(y_test, pred):.4f}")
print(f"Sai số tương đối trung vị: {np.median(np.abs(np.exp(pred) / np.exp(y_test) - 1)):.1%}")

Xem tên các feature sau tiền xử lý và hệ số của model:

In [ ]:
names = model.named_steps["preprocess"].get_feature_names_out()
coefs = pd.Series(model.named_steps["model"].coef_, index=names).sort_values()
coefs.round(3)

In [ ]:
import matplotlib.pyplot as plt

from ml_course.visualization import set_style

set_style()
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(coefs.index, coefs.values, color=np.where(coefs.values > 0, "C2", "C3"))
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("hệ số Ridge (target = log giá; feature số đã chuẩn hoá)")
ax.set_title("Hệ số theo feature sau tiền xử lý")
plt.tight_layout()
plt.show()

**Đọc hệ số** (target là log giá, feature số đã chuẩn hoá): hệ số dương lớn nhất là diện tích và các quận trung tâm; `distance` và `age` âm — khớp với cách dữ liệu được sinh ra. Đây là một lý do nên chuẩn hoá: hệ số của các feature số trở nên **so sánh được** với nhau.

## 3. Cross-validation đúng cách

Đưa **cả pipeline** vào `cross_val_score`: mỗi fold, imputer/scaler/encoder được fit lại chỉ trên phần train của fold đó.

In [ ]:
scores = cross_val_score(model, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
print(f"CV MAE: {-scores.mean():.4f} ± {scores.std():.4f}")

## 4. Tìm hyperparameter cho cả tiền xử lý lẫn model

Tham số của các bước lồng nhau được truy cập bằng `tên_bước__tên_tham_số`:

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "preprocess__num__impute__strategy": ["mean", "median"],
    "model__alpha": [0.1, 1.0, 10.0],
}
search = GridSearchCV(model, param_grid, cv=5, scoring="neg_mean_absolute_error").fit(X_train, y_train)
print("Tham số tốt nhất:", search.best_params_)
print(f"CV MAE tốt nhất: {-search.best_score_:.4f}")

## 5. Custom transformer

Khi cần một bước tiền xử lý riêng (vd: tạo feature mới), viết class kế thừa `BaseEstimator` + `TransformerMixin`:
- tham số khai báo trong `__init__` (không xử lý gì khác ở đây);
- `fit` học tham số (hậu tố `_`) và trả về `self`;
- `transform` áp dụng; thêm `get_feature_names_out` để tên cột không bị mất.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


class HouseFeatures(TransformerMixin, BaseEstimator):
    """Thêm feature: diện tích mỗi phòng ngủ, log diện tích, nhà mới (< 5 năm)."""

    def __init__(self, new_house_years=5):
        self.new_house_years = new_house_years

    def fit(self, X, y=None):
        self.feature_names_in_ = np.asarray(X.columns)
        return self

    def transform(self, X):
        X = X.copy()
        X["area_per_bedroom"] = X["area_m2"] / X["bedrooms"].clip(lower=1)
        X["log_area"] = np.log(X["area_m2"])
        X["is_new"] = (X["age_years"] < self.new_house_years).astype(float)
        return X

    def get_feature_names_out(self, input_features=None):
        return np.r_[self.feature_names_in_, ["area_per_bedroom", "log_area", "is_new"]]


model_fe = Pipeline([
    ("features", HouseFeatures()),
    ("preprocess", ColumnTransformer([
        ("num", numeric_pipe, numeric_features + ["area_per_bedroom", "log_area", "is_new"]),
        ("cat", categorical_pipe, categorical_features),
        ("bool", "passthrough", boolean_features),
    ])),
    ("model", Ridge(alpha=1.0)),
])
scores_fe = cross_val_score(model_fe, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
print(f"CV MAE không feature mới: {-scores.mean():.4f} | có feature mới: {-scores_fe.mean():.4f}")

`log_area` giúp model tuyến tính nắm được quan hệ phi tuyến giữa diện tích và log giá. Feature engineering được học sâu ở chương 17.

## 6. Chọn cột tự động và output dạng DataFrame

In [ ]:
auto = ColumnTransformer([
    ("num", StandardScaler(), make_column_selector(dtype_include="number")),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), make_column_selector(dtype_exclude="number")),
], remainder="drop").set_output(transform="pandas")
auto.fit_transform(X_train.dropna()).head(3)

## 7. Lưu và nạp pipeline

Pipeline đã fit chứa **cả** tiền xử lý và model → lưu một file duy nhất, khi triển khai chỉ cần `load` rồi `predict` trên dữ liệu thô (chương 39–40):

In [ ]:
import tempfile
from pathlib import Path

import joblib

path = Path(tempfile.gettempdir()) / "house_price_pipeline.joblib"
joblib.dump(model, path)
loaded = joblib.load(path)
new_house = pd.DataFrame([{"area_m2": 80, "bedrooms": 2, "bathrooms": np.nan, "age_years": 3, "district": "Quận 7",
                           "distance_to_center_km": 9.5, "has_garden": False}])
print(f"Giá dự đoán: {np.exp(loaded.predict(new_house))[0]:.2f} tỷ VND (bathrooms bị thiếu vẫn dự đoán được)")

## ✅ Checklist pipeline

- [ ] Chia train/test **trước** mọi bước học từ dữ liệu
- [ ] Mọi bước tiền xử lý nằm **trong** Pipeline
- [ ] Cột số: impute (+ indicator) → scale; cột phân loại: impute `"missing"` → one-hot (`handle_unknown="ignore"`)
- [ ] Cross-validation trên **toàn bộ pipeline**
- [ ] Lưu cả pipeline (không chỉ model)

## 🧠 Kiểm tra nhanh

1. Vì sao `cross_val_score(pipeline, X, y)` an toàn hơn chuẩn hoá `X` trước rồi mới cross-validate?
2. Cú pháp đặt `alpha` của bước `model` trong GridSearchCV là gì?
3. Ba quy ước khi viết custom transformer theo chuẩn sklearn?

## 📌 Tóm tắt

- `Pipeline` = chuỗi bước, `fit`/`predict` một lần; `ColumnTransformer` = xử lý song song theo nhóm cột.
- Pipeline ngăn leakage trong CV và đơn giản hoá triển khai.
- Hyperparameter của mọi bước truy cập bằng `step__param`.

## 📚 Tài liệu tham khảo

- scikit-learn — Pipelines and composite estimators: https://scikit-learn.org/stable/modules/compose.html
- scikit-learn — Developing estimators: https://scikit-learn.org/stable/developers/develop.html